# Paper reproduction: Baehr-Jones 2012 electrode (Palace)

This notebook reproduces the electrode half of Baehr-Jones *et al.*,
"Ultralow drive voltage silicon traveling-wave modulator", *Optics Express*
**20**(11), 12014 (2012), [doi:10.1364/OE.20.012014](https://doi.org/10.1364/OE.20.012014),
with generic gsim tools. The paper measured a **metal-only control line** at 10 GHz:
$Z = 61 - 5j\,\Omega$, 3.8 dB/cm, $L = 457$ nH/m, $R = 4.9$ k$\Omega$/m, $C = 0.12$ nF/m.
The junction capacitance of the modulator then loads this line, which is the
closed-form part of [the companion notebook](tcad_bj2012_modulator.ipynb).

The electrode is a pair of coplanar strips (CPS): two 50 µm wide, 2 µm thick aluminum
strips with a 5 µm gap, in oxide, above a 2 µm buried oxide on a high-resistivity wafer.

1. **A local reference.** femwell solves the cross-section in seconds, with perfect
   and with lossy aluminum.
2. **Palace set-up.** One 2D `BoundaryModeSim` per frequency (1 to 40 GHz) and two 3D
   `DrivenSim` straight sections, 200 µm and 600 µm long, with a wave port at each end.
   Everything is meshed and validated here, locally.
3. **The cloud runs.** A cell guarded by `RUN_CLOUD` submits them. It is **off** by default.
4. **Post-processing.** De-embed the short section from the long one, fit one passive
   line to the result, and compare with the paper. The whole chain is first tested on
   synthetic S-parameters of the paper's own line, with noise.

Reference values (read by eye from the paper's figures, about one marker of
uncertainty) are in `data/bj2012_reference.json`.

In [ ]:
import json
import logging
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import meshio
import numpy as np
import skfem
from scipy.constants import speed_of_light
from skfem.helpers import cross

from gsim.common.modes import Conductor, propagating_modes
from gsim.common.stack.extractor import Layer, LayerStack
from gsim.common.stack.materials import MaterialProperties
from gsim.common.transmission_line import (
    line_params_from_gamma,
    line_params_from_neff,
    loaded_line_params,
    section_abcd,
)
from gsim.femwell import epsilon_by_region, line_reading, solve_modes
from gsim.femwell.adapter import electrode_current
from gsim.palace import BoundaryModeSim, DrivenSim

# The mesher's advice on air boxes targets other simulations.
logging.getLogger("gsim.palace").setLevel(logging.ERROR)

REF_PATH = next(
    p
    for p in (
        Path("data/bj2012_reference.json"),
        Path("nbs/data/bj2012_reference.json"),
    )
    if p.exists()
)
REF = json.loads(REF_PATH.read_text(encoding="utf-8"))
PAPER = REF["metal_only_line_10GHz"]
print(
    "paper:",
    REF["citation"]["journal"],
    REF["citation"]["year"],
    REF["citation"]["doi"],
)

## The cross-section

Coordinates in µm: $y$ across the line, $z$ up, the strips along $x$. Numbers from the
paper's Fig. 1b: strips 50 µm wide and 2 µm thick with a 5 µm gap, 2 µm of buried oxide,
750 Ω·cm handle wafer (the paper's text gives only the BOX, the handle and the top-silicon
values; the electrode dimensions are read from the figure and are not checked against
anything else). **Assumptions** (the paper does not give them): the strips'
lower face is 1.5 µm above the buried oxide (the 0.09 + 0.75 + 0.6 µm of slab, lower
metal and via add up to 1.44 µm), the top silicon under the line is absent, the handle
wafer is truncated at 100 µm, and **there is no oxide over the strips**: the paper's
process description ends with "two levels of aluminum" and mentions no passivation, and a
first version of this notebook that put 2 µm of oxide over them had $C$ about 8 % and the
index 4 % higher than without it (same aluminum). The domain is $\pm 200$ µm wide with a 100 µm air layer on
top; the half-width changes $Z$ by less than 0.5 % between $\pm 200$ and $\pm 400$ µm.
The outer wall of the domain is a **perfect conductor** in the femwell solves, in J1 and in
the 3D runs: a closed metallic box, which is also what the wave port's own eigenproblem
assumes. (Section 2 gives the reason: an absorbing wall in 3D added 1 to 2.8 dB/cm to the line's loss.)

**The strips' conductivity is set from the paper's resistance.** Bulk aluminum
($3.77\times10^7$ S/m) gives 2.2 k$\Omega$/m at 10 GHz for these strips, about half the
measured 4.9 k$\Omega$/m (Section 1 shows how that number was checked), and that
is not only a loss: $L$ is the imaginary part of the series impedance over $\omega$, so it
contains the metal's internal inductance and rises with $R$ (femwell: 408 nH/m for bulk
aluminum, 436 nH/m for the effective conductivity below, against the paper's 457 nH/m).
The notebook therefore uses an **effective conductivity of $10^7$ S/m** (10 µΩ·cm), the value for which
femwell's $R$ at 10 GHz is the paper's. It stands for everything the model leaves
out (barrier layers, roughness, edge rounding, contact resistance); it is one fitted
number, and $Z$, $L$, $C$, $n$ and the loss are then predictions.

The same builder serves the 2D and 3D simulations. The background layers are
`stack.dielectrics` boxes, as in the PDK stacks; one thin slice of oxide inside the BOX
widens the layout's bounding box to the domain width, because only layers of the stack
count for it. The two strips get different GDS layer numbers because the 3D mesher
identifies conductors by layer number.

In [ ]:
STRIP_WIDTH = REF["geometry"]["electrode_strip_width_um"]
GAP = REF["geometry"]["electrode_gap_um"]
METAL_THICKNESS = REF["geometry"]["top_metal_thickness_um"]
BOX_THICKNESS = REF["geometry"]["box_thickness_um"]
HANDLE_DEPTH = 100.0
HALF_WIDTH = 200.0
AIR_ABOVE = 100.0
METAL_BOTTOM = 1.5
OXIDE_OVER_METAL = 0.0  # no passivation over the top aluminum (see above)
SIGMA_AL_BULK = 3.77e7  # S/m, bulk aluminum
SIGMA_AL = 1.0e7  # S/m, effective: femwell's R at 10 GHz is the paper's 4.9 kohm/m
SIGMA_HANDLE = 100.0 / REF["geometry"]["handle_resistivity_ohm_cm"]  # S/m
STRIPS = {  # name: (y0, y1) in um
    "strip_g": (-GAP / 2 - STRIP_WIDTH, -GAP / 2),
    "strip_s": (GAP / 2, GAP / 2 + STRIP_WIDTH),
}
STRIP_Z = (METAL_BOTTOM, METAL_BOTTOM + METAL_THICKNESS)


def build_cps(length_um=10.0, *, lossy_metal=False, ports=False, sigma=SIGMA_AL):
    """Component and layer stack of the CPS (strips along x, ``length_um`` long).

    ``lossy_metal`` meshes the strips as regions of aluminum (for femwell);
    otherwise they are conductor layers: outlines in 2D, finite-conductivity surfaces
    in 3D. ``ports`` puts a port at each end on the signal strip. ``sigma`` is the
    strips' conductivity (S/m).
    """
    gf.gpdk.PDK.activate()
    comp = gf.Component()
    layers = {}

    def add(name, gds_layer, y0, y1, z0, z1, material, layer_type):
        comp.add_polygon(
            [(0, y0), (length_um, y0), (length_um, y1), (0, y1)], layer=gds_layer
        )
        layers[name] = Layer(
            name=name,
            gds_layer=gds_layer,
            zmin=z0,
            zmax=z1,
            thickness=z1 - z0,
            material=material,
            layer_type=layer_type,
            mesh_resolution=1.0,
        )

    add("domain", (99, 0), -HALF_WIDTH, HALF_WIDTH, -1.0, -0.9, "sio2_rf", "dielectric")
    for i, (name, span) in enumerate(STRIPS.items()):
        add(
            name,
            (10 + i, 0),
            *span,
            *STRIP_Z,
            "aluminum",
            "dielectric" if lossy_metal else "conductor",
        )
    if ports:
        for name, x, orientation in (("o1", 0.0, 180), ("o2", length_um, 0)):
            comp.add_port(
                name=name,
                center=(x, sum(STRIPS["strip_s"]) / 2),
                width=STRIP_WIDTH,
                orientation=orientation,
                port_type="electrical",
                layer=(11, 0),
            )
    z_oxide_top = STRIP_Z[1] + OXIDE_OVER_METAL
    stack = LayerStack(
        pdk_name="bj2012_cps",
        layers=layers,
        materials={
            "sio2_rf": MaterialProperties(permittivity=3.9).to_dict(),
            "handle_si": MaterialProperties(
                permittivity=11.9, conductivity=SIGMA_HANDLE
            ).to_dict(),
            "aluminum": MaterialProperties(
                permittivity=1.0, conductivity=sigma
            ).to_dict(),
            "air": MaterialProperties(permittivity=1.0).to_dict(),
        },
        dielectrics=[
            {
                "name": "handle",
                "zmin": -BOX_THICKNESS - HANDLE_DEPTH,
                "zmax": -BOX_THICKNESS,
                "material": "handle_si",
            },
            {"name": "box", "zmin": -BOX_THICKNESS, "zmax": 0.0, "material": "sio2_rf"},
            {"name": "oxide", "zmin": 0.0, "zmax": z_oxide_top, "material": "sio2_rf"},
        ],
    )
    return comp, stack


def conductors(model):
    """The signal and ground strips as the current integrals name them."""
    return (
        Conductor("strip_s", (STRIPS["strip_s"], STRIP_Z), model),
        Conductor("strip_g", (STRIPS["strip_g"], STRIP_Z), model),
    )


# Refinement box in the gap and around the strip edges: (y0, y1, z0, z1, size) um.
GAP_BOX = (-10.0, 10.0, 0.8, 5.0, 0.5)


def cross_section_sim(
    lossy_metal, output_dir, *, refined_um=1.0, metal_mesh_um=1.0, sigma=SIGMA_AL
):
    """A meshed `BoundaryModeSim` of the CPS cross-section at x = 5 um."""
    comp, stack = build_cps(lossy_metal=lossy_metal, sigma=sigma)
    for name in STRIPS:
        stack.layers[name].mesh_resolution = metal_mesh_um
    sim = BoundaryModeSim()
    sim.set_output_dir(output_dir)
    sim.set_stack(stack)
    sim.set_geometry(comp)
    sim.set_airbox(margin_x=0.0, margin_y=0.0, z_above=AIR_ABOVE, material="air")
    sim.set_cross_section("x=5")
    sim.mesh(
        preset="coarse",
        refined_mesh_size=refined_um,
        max_mesh_size=20.0,
        refinement_boxes=[GAP_BOX],
        verbose=False,
    )
    return sim, stack

## Section 1: a local reference with femwell

femwell solves the shielded cross-section (perfect-conductor box) at 10 GHz in about ten
seconds per solve. Three things matter, all learned from checks on this geometry:

- the box has **two** propagating modes: the line mode between the strips, and a
  common ("wall") mode with both strips against the box. The wall mode is the slower
  one, so the default `select_line_mode` would pick it. The line mode is the slowest
  one that `line_reading` does not flag as a wall mode;
- the impedance needs **second-order** elements: $n_{eff}$ is the same at order 1 and 2, but
  the contour-current impedance moves by about 10 % between them;
- the **perfect-metal** solve has no resistance. The **lossy** solves mesh the
  aluminum (0.5 µm elements against a skin depth of 1.6 µm for the effective conductivity
  and 0.82 µm for bulk aluminum at 10 GHz) and give $R$ and the
  loss. Their impedance is read from the conduction current. Both are run, so that the
  effect of the conductivity is on the page: the cell below prints bulk aluminum next to
  the effective one;
- **the impedance depends on its definition**, and Section 4 needs three of them. `z0` is
  $Z_{PI} = 2P/|I|^2$ with the current on the signal strip, `z_vi` is $|V|/|I|$ and `z_pv` is
  $|V|^2/2P$ with $V$ along the path across the gap that J1 declares. On the perfect-metal
  mode they differ by 6 %; with lossy metal they agree to 0.5 %.

In [ ]:
FREQ_HZ = 10e9


def solve_line_mode(sim, stack, model, f_hz, *, order=2, num_modes=6):
    """Solve the box modes; return ``(reading, mode, ranking)`` of the line mode.

    ``ranking`` lists every propagating mode by decreasing index as
    ``(number, n_eff, wall_mode)``: the number a Palace wave port gives it.
    """
    signal, ground = conductors(model)
    mesh = sim.mesh_path
    modes = solve_modes(
        mesh,
        epsilon=epsilon_by_region(mesh, stack, frequency_hz=f_hz),
        wavelength_um=speed_of_light / f_hz * 1e6,
        num_modes=num_modes,
        order=order,
        metallic_boundaries=True,
        n_guess=2.3,
    )
    readings = [
        (
            line_reading(
                m, frequency_hz=f_hz, mesh=mesh, signal=signal, return_=ground
            ),
            m,
        )
        for m in propagating_modes(list(modes))
    ]
    line = [rm for rm in readings if not rm[0].wall_mode]
    best = max(line, key=lambda rm: rm[0].n_eff.real)
    by_index = sorted(readings, key=lambda rm: -rm[0].n_eff.real)
    ranking = [
        (number, rm[0].n_eff, bool(rm[0].wall_mode))
        for number, rm in enumerate(by_index, start=1)
    ]
    return (*best, ranking)


def mode_power(mode):
    """Power 1/2 Re int (E x H*)_z dA of a femwell mode, in W (mesh coordinates in um)."""

    @skfem.Functional(dtype=np.complex128)
    def form(w):
        return cross(w["E"][0], np.conj(w["H"][0]))

    basis = mode.basis
    return (
        0.5
        * form.assemble(basis, E=basis.interpolate(mode.E), H=basis.interpolate(mode.H))
        * 1e-12
    )


def gap_voltage(mode, half_um, z_um, n=400):
    """V = int E_t . dl across the gap at height z_um, from y = +half_um to -half_um (E in V/m)."""
    (e_t, e_basis), _ = mode.basis.split(mode.E)
    y = half_um - 2 * half_um * (np.arange(n) + 0.5) / n
    e_y, _e_z = e_basis.interpolator(e_t)(np.array([y, np.full(n, z_um)]))
    return complex(np.sum(e_y) * (-2 * half_um * 1e-6 / n))


def impedance_definitions(mode, reading, mesh, f_hz, signal):
    """The line mode's impedance by three definitions, in ohm.

    ``z_pi`` is gsim's ``line_reading`` (2P / |I|^2 with the contour current on the signal
    strip), ``z_vi`` is |V| / |I| and ``z_pv`` is |V|^2 / 2P, with V along the path that J1
    declares: across the gap, at mid-height of the strips, a hair inside the gap faces.
    """
    inset = 1e-3 * min(GAP, METAL_THICKNESS)
    voltage = gap_voltage(mode, GAP / 2 - inset, sum(STRIP_Z) / 2)
    current = (
        electrode_current(mode, frequency_hz=f_hz, conductor=signal, mesh=mesh) * 1e-6
    )
    power = mode_power(mode)
    return {
        "z_pi": complex(reading.z0_ohm),
        "z_vi": abs(voltage) / abs(current),
        "z_pv": abs(voltage) ** 2 / (2 * power.real),
    }


femwell_results = {}
PORT_RANKING = None  # the perfect-metal ranking: what the wave port's solve sees
# label: (lossy metal?, conductivity, output folder)
FEMWELL_CASES = {
    "perfect metal": (False, SIGMA_AL, "pec"),
    "lossy, effective sigma": (True, SIGMA_AL, "lossy"),
    "lossy, bulk aluminum": (True, SIGMA_AL_BULK, "bulk"),
}
for label, (lossy, sigma_case, folder) in FEMWELL_CASES.items():
    sim_xs, stack_xs = cross_section_sim(
        lossy,
        f"./palace-sim-bj2012-xs-{folder}",
        metal_mesh_um=0.5 if lossy else 1.0,  # 0.5 um in the lossy metal
        sigma=sigma_case,
    )
    model = "volume" if lossy else "pec"
    reading, mode, ranking = solve_line_mode(sim_xs, stack_xs, model, FREQ_HZ)
    if not lossy:
        PORT_RANKING = ranking
    lp = line_params_from_neff([FREQ_HZ], [reading.n_eff], z0_ohm=[reading.z0_ohm])
    rlgc = {k: float(v[0]) for k, v in lp.rlgc.items()}
    z_defs = impedance_definitions(
        mode, reading, sim_xs.mesh_path, FREQ_HZ, conductors(model)[0]
    )
    femwell_results[label] = {
        "n": float(lp.n_rf[0]),
        "z0": complex(reading.z0_ohm),
        "z_vi": float(z_defs["z_vi"]),
        "z_pv": float(z_defs["z_pv"]),
        "alpha_db_cm": float(lp.alpha_rf_np_m[0]) * 8.686 / 100,
        **rlgc,
    }
    print(label, {k: np.round(v, 4) for k, v in femwell_results[label].items()})

In [ ]:
paper_values = {
    "Z (ohm)": (PAPER["z0_real_ohm"], "z0"),
    "L (nH/m)": (PAPER["l_nH_per_m"], "L"),
    "C (nF/m)": (PAPER["c_nF_per_m"], "C"),
    "R (kohm/m)": (PAPER["r_kohm_per_m"], "R"),
    "alpha (dB/cm)": (PAPER["alpha_dB_per_cm"], "alpha_db_cm"),
}
scale = {"z0": 1.0, "L": 1e9, "C": 1e9, "R": 1e-3, "alpha_db_cm": 1.0}
fig, ax = plt.subplots(figsize=(8, 3.8))
width = 0.26
for k, (label, res) in enumerate(femwell_results.items()):
    ratios = []
    for paper, key in paper_values.values():
        value = res[key].real if key == "z0" else res[key]
        ratios.append(value * scale[key] / paper)
    ax.bar(np.arange(len(ratios)) + (k - 1) * width, ratios, width, label=label)
ax.axhline(1.0, color="k", lw=1)
ax.set_xticks(range(len(paper_values)), list(paper_values), fontsize=8)
ax.set_ylabel("femwell / paper (10 GHz)")
ax.set_title("Metal-only CPS: local femwell reference against the paper")
ax.legend(fontsize=8)
plt.show()

**Which mode is the wave port's mode 1?** The 3D wave port solves its own eigenproblem on
the port face with perfect conductors everywhere (the strips are listed in `WavePortPEC`, and
the outer wall is a perfect conductor too), which is the perfect-metal solve above, and Palace numbers the
modes by decreasing wave number. Mode 1 is therefore the **wall mode**, not the line mode.
The cell below lists the numbers.

In [ ]:
print("perfect-metal modes at 10 GHz, in the order a Palace wave port numbers them:")
for number, n_eff, is_wall in PORT_RANKING:
    kind = "wall mode (both strips against the box)" if is_wall else "line mode"
    print(f"  mode {number}: n_eff = {n_eff.real:.4f}, {kind}")
PORT_LINE_MODE = next(number for number, _n, is_wall in PORT_RANKING if not is_wall)
PORT_N_EFF = {number: n_eff for number, n_eff, _w in PORT_RANKING}

**Reading the three solves.** With bulk aluminum the femwell reference is 6 % below
the paper in $Z$, 11 % below in $L$ and 55 % below in $R$ (2.19 against 4.9 k$\Omega$/m), and its loss is
half the measured one. $C$ is 4 % above the paper's. The resistance is the number
that stands out, and it is not a solver problem: a power-loss estimate from the lossless
mode's fields agrees with the lossy solve to a few percent. 2.2 k$\Omega$/m is four times
the DC resistance of the two strips (0.53 k$\Omega$/m) and 3.5 times what a uniform surface
current around their perimeters would meet (0.62 k$\Omega$/m), because the current
crowds at the gap-facing corners: about half of the loss sits within 0.5 µm of them.
Refining the mesh at those sharp corners moves the power-loss estimate up by
about 10 %, not by a factor of two. To reach 4.9 k$\Omega$/m with these strips the metal must conduct about four times
worse than bulk aluminum, hence the effective conductivity of the geometry cell. The
second solve shows what that one number does: $R$ and the loss land on the paper's, and so, within
5 %, do $Z$ (59.4 - 5.0j against 61 - 5j), $n$, $L$ (436 against 457 nH/m, the internal
inductance having grown with $R$) and $C$. The Palace runs below use the effective
conductivity; they test the solver, with a 3D model and a different treatment of the conductor.

## Section 2: the Palace simulations, set up and meshed locally

| Job | What | Frequencies |
|---|---|---|
| J1 | `BoundaryModeSim`: 2D cross-section, line impedance across the gap, one job per frequency | 1, 2.5, 5, 10, 20, 40 GHz |
| J2 | `DrivenSim`: 200 µm straight section, wave port at each end, perfect-conductor outer wall | 1 to 40 GHz, 79 points |
| J3 | `DrivenSim`: 600 µm straight section, same wall | same |
| J4 (optional) | as J2 with the refinement near the conductors reduced to 0.6 µm | same |

**Why 200 and 600 µm.** The 200 µm section is both the reference and the *2x-thru*:
removing it from the 600 µm section leaves a 400 µm line. With a 100 µm difference the phase
difference at 10 GHz would be 2.7° and the loss difference 0.04 dB, too small to measure
well; 400 µm gives 11° and 0.15 dB.

**Modes.** The wave port solves its own 2D eigenproblem on the whole port face with perfect
conductors on the strips and on the outer wall, so two propagating modes exist on the
face: the **line mode** (n = 2.04) and the common "wall" mode (n = 2.36), whose index is 16 % higher. Palace
numbers them by decreasing wave number, so mode 1 is the wall mode. `WAVE_MODE` is
therefore the line mode's number from the
perfect-metal femwell solve above (2), and Section 4 checks the port's logged wave number
against that solve and the de-embedded index against J1. A first set of runs (an earlier
geometry, with an absorbing outer wall) with mode 1 gave S11 = -9.6 dB and S21 = -3.5 dB for 200 µm and a
de-embedded index of 6.4 at 5 GHz: see the caveats. For coupled arms (a GSSG layout) the terminal wave
ports of the [GSSG notebook](palace_terminal_waveport_gssg.ipynb) apply; one S-G pair needs none.

**The outer wall of the 3D domain is a perfect conductor.** The default of `DrivenSim` is an
absorbing wall (order 2). With it the 400 µm line came out 1.0 to 2.8 dB/cm lossier than
J1, the same cross-section in 2D (5.07 against 3.59 dB/cm at 10 GHz, 2.61 against 1.64 at 1 GHz,
7.66 against 4.91 at 40 GHz). With a perfect-conductor wall (the same mesh, only
the wall condition differs) the line is within 0.005 to 0.09 dB/cm of J1 from 1 to 40 GHz
(3.62 against 3.59 at 10 GHz) and the section reflects less (|S11| of the 200 µm section -40.6
against -31.4 dB at 10 GHz). The extra loss is therefore the wall condition's; why an absorbing wall
at this distance takes so much was not separated. The perfect-conductor wall is also the one the port's own
eigenproblem has. The box then carries the common wall mode in 3D as well; S shows no sign of it
(S11 and S22 agree to 3e-5, the loss is J1's). `DrivenSim` has no option for the wall, so `PECWallDrivenSim`
below sets gsim's `metallic_boundaries` switch (the one `BoundaryModeSim` has) and turns the absorbing
boundary off. A real device is not in a metal box: the box is the model of the reference solves, with
$\pm 200$ µm as its half-width (in 2D that half-width moves $Z$ by less than 0.5 % against $\pm 400$ µm; in 3D it was not varied).

No job is submitted here. The 3D mesh is the only thing that costs minutes.

In [ ]:
BM_FREQS_GHZ = (1.0, 2.5, 5.0, 10.0, 20.0, 40.0)
LENGTHS_UM = (200, 600)
DRIVEN_FMIN, DRIVEN_FMAX, DRIVEN_POINTS = 1e9, 40e9, 79  # 0.5 GHz steps
FIT_MIN_GHZ = 5.0  # the fit and its impedance anchors start here (see Section 4)
# The line mode's number on the port face (2: mode 1 is the wall mode, see above).
WAVE_MODE = PORT_LINE_MODE
# name: (length um, refinement near conductors um)
DRIVEN_CASES = {"J2": (200, 1.0), "J3": (600, 1.0), "J4": (200, 0.6)}


def boundary_mode_sim(freq_ghz):
    """J1: the 2D solve at one frequency, with the voltage path across the gap."""
    sim, _stack = cross_section_sim(False, f"./palace-sim-bj2012-2d-{freq_ghz:g}ghz")
    sim.metallic_boundaries = True  # the same perfect-conductor wall as the port solve
    sim.set_boundary_mode(
        freq=freq_ghz * 1e9, num_modes=4, save=0, target=2.3, tolerance=1e-8
    )
    z_mid = sum(STRIP_Z) / 2
    inset = 1e-3 * min(GAP, METAL_THICKNESS)  # a hair inside the gap, on the domain
    sim.add_impedance_path(
        "gap",
        voltage=[[GAP / 2 - inset, z_mid], [-(GAP / 2 - inset), z_mid]],
        nsamples=200,
    )
    sim.write_config()
    return sim


class PECWallDrivenSim(DrivenSim):
    """A `DrivenSim` whose outer wall is a perfect conductor (J1's wall and the port solve's).

    `metallic_boundaries` is the switch that puts the outer wall under the PEC boundary
    condition; `BoundaryModeSim` has it as a field, `DrivenSim` only reads it if present.
    """

    metallic_boundaries: bool = True
    absorbing_boundary: bool = False


def driven_sim(length_um, refined_um, output_dir):
    """J2-J4: a straight section with a wave port at each end and a perfect-conductor outer wall."""
    comp, stack = build_cps(length_um, ports=True)
    sim = PECWallDrivenSim()
    sim.set_output_dir(output_dir)
    sim.set_geometry(comp)
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=0.0, z_above=AIR_ABOVE, z_below=0.0)
    for port in ("o1", "o2"):
        sim.add_wave_port(
            port, layer="strip_s", max_size=True, mode=WAVE_MODE, excited=True
        )
    sim.set_driven(fmin=DRIVEN_FMIN, fmax=DRIVEN_FMAX, num_points=DRIVEN_POINTS)
    validation = sim.validate_config()
    assert validation.valid, validation.errors
    sim.mesh(
        preset="default",
        refined_mesh_size=refined_um,
        max_mesh_size=30.0,
        fmax=DRIVEN_FMAX,
        verbose=False,
    )
    sim.write_config()
    return sim


def mesh_counts(path):
    """``(tetrahedra, triangles)`` of a mesh file."""
    mesh = meshio.read(str(path))
    count = lambda kind: sum(len(c.data) for c in mesh.cells if c.type == kind)  # noqa: E731
    return count("tetra"), count("triangle")

In [ ]:
bm_sims = {ghz: boundary_mode_sim(ghz) for ghz in BM_FREQS_GHZ}
driven_sims = {
    name: driven_sim(length, refined, f"./palace-sim-bj2012-{name.lower()}")
    for name, (length, refined) in DRIVEN_CASES.items()
}

In [ ]:
print("J1 (2D cross-section, same mesh at every frequency):")
n_tri = mesh_counts(bm_sims[BM_FREQS_GHZ[0]].mesh_path)[1]
print(f"  {len(bm_sims)} jobs, {n_tri} triangles each, config written for each")
print("3D driven sections:")
for name, sim in driven_sims.items():
    tets, tris = mesh_counts(sim.mesh_path)
    length, refined = DRIVEN_CASES[name]
    print(
        f"  {name}: {length} um, refinement {refined} um: {tets} tetrahedra, "
        f"{tris} boundary triangles, validate_config valid: {sim.validate_config().valid}"
    )

## Section 3: the cloud runs

The cell below submits J1 to J3 (and J4 with `RUN_CONVERGENCE`) to the GDSFactory+
cloud, reusing cached results of identical jobs, then waits for them. Palace does not
run on every local machine, so this is a cloud step. **`RUN_CLOUD` is off by default**:
with it off the notebook runs offline and Section 4 shows the post-processing on
synthetic data. Turn it on only deliberately: whether this notebook is listed in the
documentation navigation, where it would submit these jobs on every build, is an open
decision.

In [ ]:
RUN_CLOUD = False
RUN_CONVERGENCE = False

results = {}
if RUN_CLOUD:
    import gsim
    from gsim.palace.results import load_text_results

    job_ids = {}
    for ghz, sim in bm_sims.items():
        job_ids["J1", ghz] = sim.run(wait=False, check_cache=True, verbose="status")
    for name, sim in driven_sims.items():
        if name == "J4" and not RUN_CONVERGENCE:
            continue
        job_ids[name] = sim.run(wait=False, check_cache=True, verbose="status")
    collected = dict(
        zip(
            job_ids,
            gsim.wait_for_results(list(job_ids.values()), verbose="status"),
            strict=True,
        )
    )
    results["J1"] = {
        ghz: load_text_results(collected["J1", ghz]) for ghz in BM_FREQS_GHZ
    }
    results["driven"] = {
        name: collected[name] for name in DRIVEN_CASES if name in collected
    }
else:
    print("RUN_CLOUD is off: nothing was submitted.")

## Section 4: post-processing

**De-embedding.** Both sections are simulated with the same ports, so the end effects
are the same. The 200 µm network is split into two mirrored halves with the IEEE P370
non-zero-length 2x-thru method, and removing the halves from the 600 µm network leaves
a 400 µm line. For a uniform symmetric line the ABCD matrix gives the
propagation constant $\gamma$ without branch ambiguity.

**Impedance.** $\gamma$ is exact (it is the same for any reference impedance and any end
fixture), but a wave port's S is normalized to its own mode, so `to_skrf(z0=50)` only
*assigns* 50 Ω, and the de-embedded $Z_c' = \sqrt{B/C}$ is not the line's impedance. S read in
a reference $Z_{ref}$ that is not the port's gives $Z_c' = 50\,Z_c/Z_p$: S alone fixes the ratio
of the line's impedance to the port mode's, $Z_c/Z_p = Z_c'/50$. The port mode is the
perfect-metal line mode of Section 1 (the port's own eigenproblem), so $Z_p$ comes from
femwell and $Z_c = Z_c'\,Z_p/50$ is anchored to the 3D data themselves. The cell
below checks the premise: $Z_c'/50$ should match $n/n_p$, the ratio of the de-embedded index
to the port's logged one, when the metal loss leaves $C$ unchanged.

**Which $Z_p$?** The port-mode impedance depends on its definition (the femwell cell prints
three: 58.3, 56.5 and 54.7 Ω at 10 GHz). The fit uses $Z_{PV}$, the definition for which
adding the metal loss leaves $C = n/(cZ)$ unchanged in femwell (-0.8 % at 10 GHz, against
+6.3 % for $Z_{PI}$), and the one J1 reports. The result moves with the choice (the cell prints
the others), which is a ±3 % uncertainty on $Z$, $L$ and $C$ that no S-parameter can remove.

**J1's impedance is not used as the anchor.** J1's $Z_{PV}$ is 53 Ω at 10 GHz, 10 % under
femwell's $Z_{PV}$ of the same lossy line (59.8 Ω), although the two agree on the index to 2.3 %; the
difference grows towards low frequency (21 Ω at 1 GHz against femwell's 74 Ω), where
the loss is large. With J1's values as anchors, the fit's $Z$ followed J1 (53.0 - 5.8j Ω)
and $\chi^2_{red}$ was 3.0; with the 3D data's own impedance it is 1.0. The J1 values are printed
beside the result.

One passive line
$Z = R_0 + R_s\sqrt{f}\,(1 + j) + j\omega L,\; Y = \omega C(\tan\delta + j)$ is fitted
to $\gamma$ of the 400 µm section and the anchors, with and without the internal-reactance
term, keeping the model with the lower $\chi^2$. This is the fit of the
[CPW de-embedding notebook](palace_cpw_deembedding.ipynb), kept to one section.

The functions come first, then a test on S-parameters built from the paper's own line.

In [ ]:
import skrf as rf
from scipy.optimize import least_squares
from skrf.calibration.deembedding import IEEEP370_SE_NZC_2xThru


def _positive_real_branch(z):
    return np.where(np.real(z) < 0, -z, z)


def extract_modal_parameters(net, length_m):
    """Zc, gamma and neff of a uniform symmetric two-port from its ABCD matrix."""
    a = net.a
    A, B, C, D = a[:, 0, 0], a[:, 0, 1], a[:, 1, 0], a[:, 1, 1]
    zc = _positive_real_branch(np.sqrt(B / C))
    exp_gl = (A + D) / 2 + B / zc
    gamma = (np.log(np.abs(exp_gl)) + 1j * np.unwrap(np.angle(exp_gl))) / length_m
    neff = np.imag(gamma) * speed_of_light / (2 * np.pi * net.f)
    return {"zc": zc, "gamma": gamma, "neff": neff}


def p370_extract(short_2xthru, long_fix_dut_fix, delta_length_m):
    """Remove the mirrored halves of the short section from the long one."""
    reference_z0 = float(np.real(np.median(short_2xthru.z0[:, 0])))
    deembedding = IEEEP370_SE_NZC_2xThru(
        dummy_2xthru=short_2xthru, z0=reference_z0, use_z_instead_ifft=True
    )
    dut = deembedding.deembed(long_fix_dut_fix)
    result = extract_modal_parameters(dut, delta_length_m)
    result["dut"] = dut
    return result


LINE_MODELS = ("sqrt_f", "skin")


def line_model(params, frequency_hz, model):
    """Series Z and shunt Y per metre: R0 + Rs sqrt(f/GHz) [(1 + j)] + jwL, wC (tan d + j)."""
    r0, rs, inductance, capacitance, tan_delta = params
    f = np.asarray(frequency_hz, float)
    omega = 2 * np.pi * f
    skin = rs * np.sqrt(f / 1e9)
    series = r0 + skin + 1j * omega * inductance
    if model == "skin":
        series = series + 1j * skin
    return series, omega * capacitance * (tan_delta + 1j)


def fit_line_parameters(
    frequency_hz,
    sections,
    anchor_frequency_hz,
    anchor_impedance_ohm,
    *,
    gamma_sigma_rel=1e-3,
    impedance_sigma_rel=1e-2,
    passive=True,
):
    """Fit one passive line to gamma of every ``(length_m, gamma)`` section and |Zc| anchors.

    gamma is unchanged by (R, L, G, C) -> (kR, kL, G/k, C/k): the anchors fix k only.
    Returns ``(best, fits)``; ``best`` has the lower reduced chi-square.
    """
    f = np.asarray(frequency_hz, float)
    lengths = np.array([length for length, _ in sections], float)
    gammas = np.array([gamma for _, gamma in sections])
    fa = np.asarray(anchor_frequency_hz, float)
    za = np.asarray(anchor_impedance_ohm, float)
    scale = np.array([100.0, 100.0, 1e-7, 1e-10, 1e-3])
    gamma_sigma = gamma_sigma_rel * np.median(np.abs(gammas))
    weight = lengths / lengths.max()  # an error in S gives d(gamma) ~ 1/length
    beta_per_omega = np.median(gammas.imag / (2 * np.pi * f))
    z_guess = float(np.median(za))
    rs_guess = 2 * z_guess * np.median(gammas.real / np.sqrt(f / 1e9))
    x0 = (
        np.array(
            [0.0, rs_guess, beta_per_omega * z_guess, beta_per_omega / z_guess, 0.0]
        )
        / scale
    )
    fits = {}
    for model in LINE_MODELS:

        def residuals(x, model=model):
            series, shunt = line_model(x * scale, f, model)
            d = (np.sqrt(series * shunt)[None] - gammas) * weight[:, None] / gamma_sigma
            za_series, za_shunt = line_model(x * scale, fa, model)
            dz = (
                np.log(np.abs(np.sqrt(za_series / za_shunt)) / za) / impedance_sigma_rel
            )
            return np.concatenate([d.real.ravel(), d.imag.ravel(), dz])

        if passive:
            # R0, Rs, L, C and tan_delta are all non-negative for a passive line. Without
            # the bound R and tan_delta trade places: both add loss, only R is large.
            res = least_squares(
                residuals,
                np.maximum(x0, 1e-6),
                method="trf",
                bounds=(0.0, np.inf),
                xtol=1e-15,
                ftol=1e-15,
                gtol=1e-15,
            )
        else:  # unbounded: a negative tan_delta then shows up instead of being clipped
            res = least_squares(
                residuals, x0, method="lm", xtol=1e-15, ftol=1e-15, gtol=1e-15
            )
        dof = max(res.fun.size - res.x.size, 1)
        names = ("R0", "Rs", "L", "C", "tan_delta")
        fits[model] = {
            "params": dict(zip(names, res.x * scale, strict=True)),
            "chi2_red": float(res.fun @ res.fun / dof),
            "model": model,
        }
    return min(fits.values(), key=lambda fit: fit["chi2_red"]), fits


def line_at(fit, f_hz):
    """gamma, Zc and RLGC of a fitted line at one frequency."""
    series, shunt = line_model(list(fit["params"].values()), [f_hz], fit["model"])
    series, shunt = series[0], shunt[0]
    omega = 2 * np.pi * f_hz
    return {
        "gamma": np.sqrt(series * shunt),
        "z0": np.sqrt(series / shunt),
        "R": series.real,
        "L": series.imag / omega,
        "G": shunt.real,
        "C": shunt.imag / omega,
    }


def summarize(line):
    """Z0, n, alpha (dB/cm) and RLGC of ``line_at``, in the paper's units."""
    gamma = line["gamma"]
    return {
        "Z0 (ohm)": line["z0"],
        "n_RF": gamma.imag / (2 * np.pi * FREQ_HZ) * speed_of_light,
        "alpha (dB/cm)": gamma.real * 8.686 / 100,
        "L (nH/m)": line["L"] * 1e9,
        "R (kohm/m)": line["R"] * 1e-3,
        "C (nF/m)": line["C"] * 1e9,
        "G (S/m)": line["G"],
    }


def anchors_in_band(anchor_frequency_hz, anchor_impedance_ohm, fit_min_ghz):
    """The impedance anchors at or above the lowest frequency of the fit."""
    fa = np.asarray(anchor_frequency_hz, float)
    za = np.asarray(anchor_impedance_ohm, float)
    keep = fa >= fit_min_ghz * 1e9 * (1 - 1e-9)
    if not keep.any():
        raise ValueError(f"no impedance anchor at or above {fit_min_ghz} GHz")
    return fa[keep], za[keep]


def deembed_and_fit(
    network_short,
    network_long,
    *,
    lengths_um=LENGTHS_UM,
    anchor_frequency_hz=None,
    anchor_impedance_ohm=None,
    fit_min_ghz=5.0,
    anchor_band_ghz=(5.0, 20.0),
    port_impedance_ohm=None,
):
    """De-embed the short section from the long one and fit the line.

    Without anchors the impedance comes from the de-embedded Zc itself, over
    ``anchor_band_ghz``. ``port_impedance_ohm`` is the impedance the ports' S refers to:
    S read in the network's reference ``z_ref`` gives ``Zc' = z_ref Zc / Zp``, so
    ``Zc = Zc' Zp / z_ref`` at about eight frequencies of the band. Without it S is taken
    to refer to the network's reference (a lumped port's S does) and the band-mean
    magnitude is the anchor.
    """
    delta_m = (lengths_um[1] - lengths_um[0]) * 1e-6
    extracted = p370_extract(network_short, network_long, delta_m)
    f = extracted["dut"].f
    mask = f >= fit_min_ghz * 1e9
    if anchor_frequency_hz is None:
        band = (f >= anchor_band_ghz[0] * 1e9) & (f <= anchor_band_ghz[1] * 1e9)
        if port_impedance_ohm is None:
            anchor_frequency_hz = f[band]
            anchor_impedance_ohm = np.full(
                band.sum(), np.abs(extracted["zc"][band]).mean()
            )
        else:
            z_ref = float(np.real(np.median(network_short.z0[:, 0])))
            in_band = np.flatnonzero(band)
            pick = in_band[
                np.unique(np.linspace(0, in_band.size - 1, 8).round().astype(int))
            ]
            anchor_frequency_hz = f[pick]
            anchor_impedance_ohm = (
                np.abs(extracted["zc"][pick]) * port_impedance_ohm / z_ref
            )
    best, fits = fit_line_parameters(
        f[mask],
        [(delta_m, extracted["gamma"][mask])],
        anchor_frequency_hz,
        anchor_impedance_ohm,
    )
    return {"extracted": extracted, "fit": best, "fits": fits, "frequency_hz": f}

### A test on the paper's own line

Exact S-parameters of a line with the paper's metal-only values at 10 GHz, built
without the extraction code: $\gamma = (R + j\omega L)/Z_0$ with $Z_0 = 61 - 5j\,\Omega$,
$R = 4.9$ k$\Omega$/m, $L = 457$ nH/m gives the shunt admittance; $R$ scales as $\sqrt{f}$, $L$ and $C$
are constant and the conductance is $\omega C \tan\delta$. Each section gets a
series-inductance and shunt-capacitance fixture at both ends (10 pH, 10 fF); the ABCD
matrices are multiplied directly and converted to S in a real reference by the textbook
formula. Two references: S in 50 Ω (what a lumped port gives; the anchors come
from the de-embedded $|Z_c|$), and S in a 61 Ω reference but *assigned* 50 Ω, as a wave
port's power-normalized S is. For that one the impedance comes either from a "2D solve" with
1 % noise or, as for the Palace data below, from the de-embedded $|Z_c'|$ and the port's
reference (61 Ω, taken as known). Noise
of $10^{-3}$ is added to every S-parameter. Recovery is checked at 10 GHz.

In [ ]:
FREQ_GRID_HZ = np.arange(2, 81) * 0.5e9  # 1 to 40 GHz, 79 points


def true_line(f_hz):
    """Per-metre R, L, G, C of the paper's line at ``f_hz`` (R ~ sqrt f)."""
    omega10 = 2 * np.pi * 10e9
    z0 = PAPER["z0_real_ohm"] + 1j * PAPER["z0_imag_ohm"]
    r10, inductance = PAPER["r_kohm_per_m"] * 1e3, PAPER["l_nH_per_m"] * 1e-9
    gamma10 = (r10 + 1j * omega10 * inductance) / z0
    y10 = gamma10 / z0
    capacitance, tan_delta = y10.imag / omega10, y10.real / y10.imag
    f = np.asarray(f_hz, float)
    omega = 2 * np.pi * f
    return {
        "R": r10 * np.sqrt(f / 10e9),
        "L": np.full(f.shape, inductance),
        "C": np.full(f.shape, capacitance),
        "G": omega * capacitance * tan_delta,
    }


def true_gamma_z0(f_hz):
    rlgc = true_line(f_hz)
    omega = 2 * np.pi * np.asarray(f_hz, float)
    series = rlgc["R"] + 1j * omega * rlgc["L"]
    shunt = rlgc["G"] + 1j * omega * rlgc["C"]
    return np.sqrt(series * shunt), np.sqrt(series / shunt)


def abcd_to_s(abcd, z_ref):
    """S-parameters of a two-port from its ABCD matrix, real reference ``z_ref``."""
    a, b, c, d = abcd[..., 0, 0], abcd[..., 0, 1], abcd[..., 1, 0], abcd[..., 1, 1]
    den = a + b / z_ref + c * z_ref + d
    s = np.empty(abcd.shape, dtype=complex)
    s[..., 0, 0] = (a + b / z_ref - c * z_ref - d) / den
    s[..., 0, 1] = 2 * (a * d - b * c) / den
    s[..., 1, 0] = 2 / den
    s[..., 1, 1] = (-a + b / z_ref - c * z_ref + d) / den
    return s


def synthetic_network(
    length_um, *, reference_ohm, noise, rng, fixture=(10e-12, 10e-15)
):
    """Noisy S of fixture + line + mirrored fixture, labelled with a 50 ohm reference."""
    f = FREQ_GRID_HZ
    omega = 2 * np.pi * f
    gamma, z0 = true_gamma_z0(f)
    line = section_abcd(gamma * length_um * 1e-6, z0)
    ones, zeros = np.ones_like(f), np.zeros_like(f)
    series_l = np.stack(
        [np.stack([ones, 1j * omega * fixture[0]], -1), np.stack([zeros, ones], -1)], -2
    )
    shunt_c = np.stack(
        [np.stack([ones, zeros], -1), np.stack([1j * omega * fixture[1], ones], -1)], -2
    )
    launch = series_l @ shunt_c
    launch_mirrored = shunt_c @ series_l
    s = abcd_to_s(launch @ line @ launch_mirrored, reference_ohm)
    s = s + noise * (rng.standard_normal(s.shape) + 1j * rng.standard_normal(s.shape))
    s = 0.5 * (s + np.transpose(s, (0, 2, 1)))
    return rf.Network(frequency=rf.Frequency.from_f(f, unit="Hz"), s=s, z0=50.0)


def recover(
    reference_ohm, *, noise=1e-3, seed=7, anchored=False, port_impedance_ohm=None
):
    """Run the de-embedding and fit on synthetic S; return (summary, truth summary).

    ``anchored`` takes the impedance from a noisy "2D solve"; ``port_impedance_ohm`` takes it
    from the de-embedded Zc and the port reference given (what the notebook does for Palace).
    """
    rng = np.random.default_rng(seed)
    short, long_ = (
        synthetic_network(length, reference_ohm=reference_ohm, noise=noise, rng=rng)
        for length in LENGTHS_UM
    )
    kwargs = {}
    if anchored:  # a "2D solve": |Z0| at a few frequencies, 1 % noise
        fa = np.array([5, 10, 20, 40]) * 1e9
        _, z_true = true_gamma_z0(fa)
        kwargs = {
            "anchor_frequency_hz": fa,
            "anchor_impedance_ohm": np.abs(z_true)
            * (1 + 0.01 * rng.standard_normal(fa.size)),
        }
    out = deembed_and_fit(short, long_, port_impedance_ohm=port_impedance_ohm, **kwargs)
    found = summarize(line_at(out["fit"], FREQ_HZ))
    rlgc = true_line([FREQ_HZ])
    gamma, z0 = true_gamma_z0([FREQ_HZ])
    truth = summarize(
        {
            "gamma": gamma[0],
            "z0": z0[0],
            "R": rlgc["R"][0],
            "L": rlgc["L"][0],
            "C": rlgc["C"][0],
            "G": rlgc["G"][0],
        }
    )
    return found, truth, out


synthetic = {
    "S in 50 ohm, anchors from |Zc|": recover(50.0),
    "S in 61 ohm assigned 50, 2D anchors": recover(61.0, anchored=True),
    "S in 61 ohm assigned 50, port reference 61": recover(
        61.0, port_impedance_ohm=61.0
    ),
}

In [ ]:
for case, (found, truth, out) in synthetic.items():
    print(
        f"{case}  (model: {out['fit']['model']}, chi2_red {out['fit']['chi2_red']:.2f})"
    )
    print(f"  {'quantity':14s} {'truth':>18s} {'recovered':>18s} {'error':>8s}")
    worst = 0.0
    for key, value in truth.items():
        got = found[key]
        err = abs(got - value) / abs(value)
        if key != "G (S/m)":  # G is a small correction: shown, not asserted
            worst = max(worst, err)
        print(f"  {key:14s} {value:18.4f} {got:18.4f} {err:8.2%}")
    print(f"  worst error (excluding G): {worst:.2%}")

In [ ]:
# The same test over eight noise seeds: how much of the error is noise.
stats = {case: [] for case in ("S in 50 ohm", "S in 61 ohm, 2D anchors")}
for seed in range(1, 9):
    for case, (ref, anch) in zip(stats, ((50.0, False), (61.0, True)), strict=True):
        found, truth, _out = recover(ref, seed=seed, anchored=anch)
        stats[case].append(
            {
                k: abs(found[k] - truth[k]) / abs(truth[k])
                for k in truth
                if k != "G (S/m)"
            }
        )
for case, rows in stats.items():
    print(case + ": worst / median error over 8 noise seeds")
    print(
        "  "
        + "  ".join(
            f"{k.split()[0]} {max(r[k] for r in rows):.1%}/{np.median([r[k] for r in rows]):.1%}"
            for k in rows[0]
        )
    )

**Reading the test.** With 79 frequency points the index, impedance, inductance,
capacitance and loss come back within about 2 % in every seed. The resistance is the weak
one: $R = \mathrm{Re}(\gamma Z_c)$ is the small real part of a mostly reactive
impedance, it trades places with the conductance in the fit (the fit is bounded to a
passive line for that reason), and 10$^{-3}$ noise on a 400 µm section leaves it about 2 %
off typically and 5 % off in the worst of eight seeds. With only 40 points the worst
seeds were 4 % in $Z_c$ and 20 % in $R$, which is why the sweep has 79 points. The loss in
dB/cm is the better-determined number for comparing with the paper. Zero noise leaves a
1 % bias in $Z_c$ from the 2x-thru split.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
gamma_true, z_true = true_gamma_z0(FREQ_GRID_HZ)
for case, (_found, _truth, out) in synthetic.items():
    f = out["frequency_hz"] / 1e9
    axes[0].plot(f, out["extracted"]["gamma"].real * 8.686 / 100, label=case)
    axes[1].plot(
        f,
        out["extracted"]["gamma"].imag
        * speed_of_light
        / (2 * np.pi * out["frequency_hz"]),
    )
axes[0].plot(FREQ_GRID_HZ / 1e9, gamma_true.real * 8.686 / 100, "k--", label="truth")
axes[1].plot(
    FREQ_GRID_HZ / 1e9,
    gamma_true.imag * speed_of_light / (2 * np.pi * FREQ_GRID_HZ),
    "k--",
)
axes[0].set(
    xlabel="f (GHz)", ylabel="alpha (dB/cm)", title="De-embedded 400 um section"
)
axes[1].set(xlabel="f (GHz)", ylabel="n_RF", title="")
axes[0].legend(fontsize=7)
plt.show()

### The Palace results

With `RUN_CLOUD` on, the cell below checks the port mode against J1, runs the
same de-embedding and fit on J2 and J3 (the impedance from the de-embedded $Z_c'$ and the
port-mode impedance, see Section 4), and prints the comparison with the paper and with the
femwell reference. With it off it says so and the summary table below shows the femwell
reference alone. It also prints what the choices cost: the fit for each definition of the
port impedance, the same fit anchored on J1's $Z_{PV}$ as the first version did, and the
loss read straight from the S data beside J1's.

The line mode of each J1 solve is the mode with the largest gap voltage: on the common mode both
strips sit at one potential and the gap carries almost none.

In [ ]:
palace = None
palace_summary = None
if results.get("J1") and len(results.get("driven", {})) >= 2:
    anchors = {}
    for ghz, res in results["J1"].items():
        voltages = {m: abs(res.mode_voltage(index=1, mode=m) or 0.0) for m in res.modes}
        mode = max(voltages, key=voltages.get)
        anchors[ghz] = (
            res.modes[mode]["n_eff"],
            res.characteristic_impedance(index=1, mode=mode),
        )
    fa = np.array(list(anchors)) * 1e9
    za = np.array([anchors[g][1] for g in anchors])
    fa, za = anchors_in_band(fa, za, FIT_MIN_GHZ)
    # The port solve must have carried the mode we asked for: its logged wave number against
    # the perfect-metal femwell solve (the same eigenproblem), and the continuity of the mode
    # over the sweep (gsim's own check of the log).
    from gsim.palace.validation import ModeExpectation, validate_solution

    k0_10 = 2 * np.pi * FREQ_HZ / speed_of_light
    for name in ("J2", "J3"):
        log_path = next(
            (
                Path(p)
                for key, p in (
                    getattr(results["driven"][name], "files", None) or {}
                ).items()
                if Path(key).name == "palace.log"
            ),
            None,
        )
        if log_path is None:
            print(
                f"{name}: no palace.log among the downloaded files, port check skipped"
            )
            continue
        report = validate_solution(
            log_path.read_text(encoding="utf-8", errors="replace"),
            mode_expectations=[
                ModeExpectation(
                    port=1,
                    mode=WAVE_MODE,
                    wave_number=complex(PORT_N_EFF[WAVE_MODE]) * k0_10,
                    wave_number_rtol=0.01,
                    frequency_ghz=FREQ_HZ / 1e9,
                )
            ],
            ports_share_cross_section=True,
        )
        for check in (*report.mode_checks, *report.port_mode_checks):
            print(f"{name} {check.name}: {check.status}; {check.evidence[0]}")
        assert not report.failures, (
            f"{name}: the wave port did not carry mode {WAVE_MODE} of the perfect-metal "
            f"solve (n = {PORT_N_EFF[WAVE_MODE].real:.4f} at 10 GHz): {report.failures}"
        )
    networks = {name: results["driven"][name].to_skrf(z0=50.0) for name in ("J2", "J3")}
    # The ports' S refers to the perfect-metal line mode (the port's own eigenproblem), not to the
    # 50 ohm that to_skrf assigns: Zc = Zc' Zp / 50 with Zp from femwell (Z_PV, see above).
    perfect = femwell_results["perfect metal"]
    palace = deembed_and_fit(
        networks["J2"],
        networks["J3"],
        port_impedance_ohm=perfect["z_pv"],
        fit_min_ghz=FIT_MIN_GHZ,
    )
    # The line must be the line mode: compare gamma with J1's complex index.
    # Above 5 GHz: the 400 um section's phase is too small below it to be
    # read to a percent. The wall mode sits well above the line mode in index.
    extracted = palace["extracted"]
    k0s = 2 * np.pi * palace["frequency_hz"] / speed_of_light
    print(
        "GHz | n_RF driven, 2D | alpha driven (raw, no fit), 2D (dB/cm) | |Zc'| x Zp/50, J1 Z_PV (ohm)"
    )
    for ghz, (n_eff, z_pv_j1) in anchors.items():
        if ghz < 5.0:
            continue
        k = int(np.argmin(np.abs(palace["frequency_hz"] - ghz * 1e9)))
        n_driven = extracted["neff"][k]
        alpha_raw = extracted["gamma"][k].real * 8.686 / 100
        alpha_2d = -n_eff.imag * k0s[k] * 8.686 / 100
        z_3d = abs(extracted["zc"][k]) * perfect["z_pv"] / 50.0
        print(
            f"{ghz:5.1f} | {n_driven:.4f}, {n_eff.real:.4f} | {alpha_raw:.3f}, {alpha_2d:.3f} | "
            f"{z_3d:.2f}, {z_pv_j1:.2f}"
        )
        assert abs(n_driven / n_eff.real - 1) < 0.04, (
            "the de-embedded index is not the line mode's: check WAVE_MODE and the "
            "port wave numbers printed above"
        )
    # The premise of the port reference: Zc'/50 = Zc/Zp should follow n/n_p (a loss that leaves
    # C unchanged). Printed, not asserted: it is a check on the story, not on the data.
    k10 = int(np.argmin(np.abs(palace["frequency_hz"] - FREQ_HZ)))
    ratio_z = abs(extracted["zc"][k10]) / 50.0
    ratio_n = extracted["neff"][k10] / PORT_N_EFF[WAVE_MODE].real
    print(
        f"10 GHz: |Zc'|/50 = {ratio_z:.3f} against n / n_port = {ratio_n:.3f} "
        f"({'consistent' if abs(ratio_z / ratio_n - 1) < 0.03 else 'CHECK the port reference'})"
    )
    # How much the definition of the port impedance moves the result at 10 GHz.
    print("fit at 10 GHz for each definition of the port-mode impedance Zp:")
    for label, key in (("Z_PI", "z0"), ("Z_VI", "z_vi"), ("Z_PV", "z_pv")):
        zp = complex(perfect[key]).real
        alt = summarize(
            line_at(
                deembed_and_fit(
                    networks["J2"],
                    networks["J3"],
                    port_impedance_ohm=zp,
                    fit_min_ghz=FIT_MIN_GHZ,
                )["fit"],
                FREQ_HZ,
            )
        )
        z = alt["Z0 (ohm)"]
        print(
            f"  Zp = {label} = {zp:.2f}: Z = {z.real:.1f}{z.imag:+.1f}j ohm, "
            f"L = {alt['L (nH/m)']:.0f} nH/m, C = {alt['C (nF/m)']:.4f} nF/m, R = {alt['R (kohm/m)']:.2f} kohm/m"
        )
    # For comparison, the first version: J1's Z_PV as the anchors.
    by_j1 = deembed_and_fit(
        networks["J2"],
        networks["J3"],
        anchor_frequency_hz=fa,
        anchor_impedance_ohm=za,
        fit_min_ghz=FIT_MIN_GHZ,
    )
    j1_row = summarize(line_at(by_j1["fit"], FREQ_HZ))
    print(
        f"anchored on J1's Z_PV instead (chi2_red {by_j1['fit']['chi2_red']:.1f} against "
        f"{palace['fit']['chi2_red']:.1f}): Z = {j1_row['Z0 (ohm)']:.1f} ohm, "
        f"L = {j1_row['L (nH/m)']:.0f} nH/m, C = {j1_row['C (nF/m)']:.4f} nF/m"
    )
    palace_summary = summarize(line_at(palace["fit"], FREQ_HZ))
    print({k: np.round(v, 4) for k, v in palace_summary.items()})
else:
    print("No Palace results (RUN_CLOUD is off): nothing to extract.")

In [ ]:
paper_row = {
    "Z0 (ohm)": complex(PAPER["z0_real_ohm"], PAPER["z0_imag_ohm"]),
    "n_RF": None,
    "alpha (dB/cm)": PAPER["alpha_dB_per_cm"],
    "L (nH/m)": PAPER["l_nH_per_m"],
    "R (kohm/m)": PAPER["r_kohm_per_m"],
    "C (nF/m)": PAPER["c_nF_per_m"],
}
femwell_lossy = femwell_results["lossy, effective sigma"]
femwell_row = {
    "Z0 (ohm)": femwell_lossy["z0"],
    "n_RF": femwell_lossy["n"],
    "alpha (dB/cm)": femwell_lossy["alpha_db_cm"],
    "L (nH/m)": femwell_lossy["L"] * 1e9,
    "R (kohm/m)": femwell_lossy["R"] * 1e-3,
    "C (nF/m)": femwell_lossy["C"] * 1e9,
}
tolerance = {  # plan tolerances for the metal-only line
    "Z0 (ohm)": 0.10,
    "alpha (dB/cm)": 0.30,
    "L (nH/m)": 0.10,
    "R (kohm/m)": 0.30,
    "C (nF/m)": 0.10,
}
print(
    f"{'quantity':14s} {'paper':>14s} {'femwell':>14s} {'Palace':>14s}  error / tolerance (Palace if run, else femwell)"
)
for key, paper_value in paper_row.items():
    if paper_value is None:
        continue
    fem = femwell_row[key]
    pal = palace_summary[key] if palace else None
    simulated = pal if pal is not None else fem  # Palace once it has run
    err = abs(simulated - paper_value) / abs(paper_value)
    flag = "ok" if err <= tolerance[key] else "OUTSIDE"
    cells = [
        np.round(value, 3) if value is not None else "(cloud)"
        for value in (paper_value, fem, pal)
    ]
    print(
        f"{key:14s} {cells[0]!s:>14} {cells[1]!s:>14} {cells[2]!s:>14}"
        f"  {err:6.1%} / {tolerance[key]:.0%} {flag}"
    )

## The loaded line

The paper's loaded line at 10 GHz: $Z = 37 - 1j\,\Omega$, 10.8 dB/cm, $n_{RF} = 3.75$. The
modulator's junction adds a series-RC branch per unit length to the shunt admittance of the
metal-only line; the junction values are those of the companion notebook at 0 V (calibrated
TCAD, averaged over the striated length: $R_s = 1.50$ Ω·cm, $C_j = 0.222$ nF/m) and the paper's quoted ones (1.6 Ω·cm, 0.225 nF/m). The
metal-only RLGC is the Palace fit when it exists and the femwell reference otherwise.

In [ ]:
JUNCTIONS = {
    "paper-quoted junction": (0.016, 0.225e-9),
    "calibrated TCAD junction": (0.0150, 0.222e-9),
}
if palace:
    line10 = line_at(palace["fit"], FREQ_HZ)
    source = "Palace"
    rlgc10 = {k: np.array([line10[k]]) for k in "RLGC"}
else:
    source = "femwell (lossy, effective sigma)"
    rlgc10 = {k: np.array([femwell_lossy[k]]) for k in "RLGC"}
for label, junction in JUNCTIONS.items():
    gamma_l, z_l = loaded_line_params([FREQ_HZ], rlgc=rlgc10, junction=junction)
    lp = line_params_from_gamma([FREQ_HZ], gamma_l, z0_ohm=z_l)
    print(
        f"{source} + {label}: Z = {z_l[0].real:.1f}{z_l[0].imag:+.1f}j ohm, "
        f"n_RF = {lp.n_rf[0]:.3f}, alpha = {lp.alpha_rf_np_m[0] * 8.686 / 100:.2f} dB/cm"
    )
print("paper: Z = 37-1j ohm, n_RF = 3.75, alpha = 10.8 dB/cm")

## What is and is not established here

- The electrode dimensions are read from a figure, and the paper's text gives none of
  them; the absence of oxide over the strips, the height of
  the strips above the BOX, the absent top silicon and the truncation of the domain are
  assumptions. The oxide cover alone moves $n_{RF}$ by about 4 % and $C$ by about 8 %.
  A different set of figure-reading errors would also bring the perfect-metal $L$ and $C$ to
  the paper's (a gap of 9 µm instead of 5, with the strips 0.75 µm above the BOX, gives
  458 nH/m and 0.118 nF/m); nothing in the paper supports those numbers, so the notebook
  does not use them, and such a change does not raise $R$.
- **The effective conductivity is a fit, not a measurement of the metal.** It is chosen so
  that femwell's resistance at 10 GHz is the paper's; the impedance, inductance, capacitance, index
  and loss that then agree with the paper (to 5 %) are not independent of it, because
  $L$ contains the internal inductance, which grows with $R$, and the loss is mostly $R/2Z$. With bulk
  aluminum the same geometry gives half the resistance and loss. Whether the real metal
  is four times less conductive than bulk aluminum, or the extra loss has another origin, is not
  something this notebook can say.
- The Palace runs test the solver against femwell on the same geometry. Palace's `Thickness`
  option on a finite-conductivity boundary is used with a skin depth (1.6 µm) close to the
  strip thickness (2 µm), where how a surface-impedance model handles a conductor
  seen from both sides may matter; femwell meshes the metal. A difference between the two there
  says something about the surface-impedance model, not about the paper.
- The line impedance depends on how it is defined. J1 reports $Z_{PV} = |V|^2 / 2P$ with $V$ along
  a path across the gap, femwell's `line_reading` $Z_{PI} = 2P / |I|^2$ with $I$ on
  the signal strip. For the lossy solve they agree to 0.5 %
  (59.4 and 59.8 ohm). On the perfect-metal mode of the same mesh they differ by about 6 % (58.3 and
  54.7 ohm; $|V|/|I|$ is 56.5), so a comparison of a Palace $Z_{PV}$ with a perfect-metal femwell
  $Z_{PI}$ carries that difference. For the 3D result it is the main uncertainty left: with the
  port reference taken as $Z_{PI}$, $Z_{VI}$ or $Z_{PV}$ the 3D line is 61.8 - 5.6j, 59.9 - 5.4j or
  58.1 - 5.2j Ω (the cell prints it); the notebook reports the last.
- **J1's $Z_{PV}$ disagrees with femwell's for the same line.** femwell's lossy $Z_{PV}$ is 59.8 Ω at
  10 GHz and J1's is 53.4 Ω; at 1 GHz 74 against 21 Ω, at 40 GHz 57.2 against
  55.9 Ω, while the index agrees to 2.3 % and the loss to 4 % up to 10 GHz. The 3D S data read with
  the port reference above (real part of $Z_c' Z_{p,PV}/50$) sit within 3.5 % of femwell's lossy $Z_{PV}$ from
  5 to 40 GHz (60.1, 58.0, 57.3, 58.2 Ω against 62.2, 59.8, 58.2, 57.2 Ω at 5, 10, 20, 40 GHz) and 4 % and 7 % below it at
  2.5 and 1 GHz (63.2 and 69.1 Ω against 66.0 and 74.4 Ω), a comparison that
  rests on the port-reference premise. An impedance anchored on J1 (the first version of this cell)
  came out at 53 Ω because it followed J1, not because the 3D run is off. Why J1's $Z_{PV}$ is low was
  not found (its post-processing is Palace's, and the 2D mesh has no cells inside the strips where the 3D mesh has
  oxide). Two 2D runs narrow it down: with the absorbing wall instead of the perfect conductor $Z_{PV}$ at 1 GHz is
  21.0 Ω (J1: 21.3), so the wall is not the cause; with the conductor surfaces perfect (the port's own eigenproblem)
  Palace's $Z_{PV}$ is 54.66 Ω at 10 GHz against femwell's 54.71 Ω, with the same index to six digits, so the
  low value comes with the finite-conductivity surface. J1's index and loss serve as comparison; its impedance is not used.
- **The 3D loss agrees with J1 because the outer wall is metallic; with an absorbing wall it did not.** The
  first 3D runs used the default absorbing wall (order 2). The loss from the eigenvalues of
  their S data (no fit, no impedance) was 5.07 dB/cm at 10 GHz (the fit's 5.01, 32 % above the paper's 3.8,
  outside the 30 % tolerance), J1's 3.59 and femwell's 3.74, and 0.97 to 2.75 dB/cm above J1 at each of the six
  frequencies from 1 to 40 GHz. The same meshes with a perfect-conductor outer wall give 3.62 dB/cm at
  10 GHz from the eigenvalues (the fit's 3.56, 6 % under the paper's) and are within 0.005 to 0.094 dB/cm of J1 at
  all six frequencies (0.3 % at 1 GHz, 0.8 % at 10 GHz, 1.9 % at 40 GHz), with the index within 0.3 %: 97 to 99 % of the excess
  is the wall. In 2D the same wall change (order-2 absorbing wall instead of the perfect conductor) gives 0.02, 0.46, 1.08 and
  2.73 dB/cm at 1, 10, 20 and 40 GHz, which is 2, 31, 60 and 99 % of the 3D excess: the 3D wall costs more than the
  2D one below 20 GHz, and why was not separated. A wider absorbing domain (±400 µm) did not help: its wave-port mode solve
  did not converge at most frequencies. A twice coarser mesh on the 2D metal outline changes the loss by 0.05 %, so the edge
  mesh is not the cause, as far as 2D shows. What is left, a small excess over J1 that grows with frequency, is
  not assigned: the strips differ (a hole bounded by an exterior surface in 2D; an oxide-filled shell, an interior
  boundary with the conductor condition on both faces, in 3D), and so do the 3D mesh and the de-embedding. It is not a check against
  an open device: J1, femwell and the 3D runs share the metallic box, and the box's half-width was not varied in 3D.
- The post-processing is tested on synthetic S-parameters with noise, not on a Palace
  result. Which mode the wave port carries is checked at run time, against the
  perfect-metal femwell solve (the port's own eigenproblem) and against J1.
- **The wave-port mode number matters.** A first set of J2/J3 runs (an earlier geometry)
  used `mode=1`. On this face that is the wall mode (n = 2.38 in the port log, 2.382 in
  the femwell perfect-metal solve of that geometry), and a wave in it needs the metallic outer wall that only the port
  solve had (the 3D domain's wall was absorbing then; mode 1 was not run again with the metallic wall of this
  version): S11 was -9.6 dB and S21 -3.5 dB for 200 µm, nearly flat
  over 1 to 40 GHz, S21 -7 to -9 dB for 600 µm, and the de-embedded "index" was 6.4 at
  5 GHz against J1's 2.21. Neither the
  reduced wave-port model nor the 50 Ω assigned by `to_skrf` was the cause: the port mode
  did not change over the sweep, and $\gamma$ of a symmetric line does not depend on the
  reference impedance given to equal ports (only the extracted $|Z_c|$ scales with it, so
  the impedance has to come from the port's own reference, see Section 4). The line mode is port mode 2.
- The port eigenproblem treats the strips as perfect conductors, the 3D problem as aluminum
  with a surface impedance: the port's line mode (n = 2.04, 58 Ω) is not exactly the
  physical line's (femwell, lossy: n = 2.22, 59 - 5j Ω at 10 GHz). The difference is a launch mismatch that the
  2x-thru de-embedding removes; whether it does to the accuracy needed is checked by the
  index and impedance comparison with J1, not assumed.